# Understanding the Pipeline / パイプラインの理解

`run_pipeline()` orchestrates the entire workflow: selecting fonts, filtering characters,
rendering images, and writing datasets. This notebook covers configuration, execution, and result interpretation.

In [ ]:
import logging
from pathlib import Path
from font2dataset.pipeline import (
    PipelineConfig, run_pipeline,
)
from font2dataset.renderer import RenderConfig

# Enable debug logging to see pipeline progress
logging.basicConfig(
    level=logging.DEBUG,
    format='%(name)s [%(levelname)s] %(message)s'
)

print("Ready")

## 1. Configuration / 設定

Pipeline is driven entirely by `PipelineConfig`.

In [ ]:
# Basic configuration: ASCII characters from local fonts
config = PipelineConfig(
    charset="digits",  # preset name
    font_dir="./fonts",
    output_dir="./test_pipeline_output",
    render=RenderConfig(
        image_size=(64, 64),
        font_size=48,
        background="white",
        foreground="black",
    ),
    workers=2,  # 2 parallel worker threads
)

print(f"Charset: {config.charset}")
print(f"Font dir: {config.font_dir}")
print(f"Output dir: {config.output_dir}")
print(f"Workers: {config.workers}")
print(f"Image size: {config.render.image_size}")
print(f"Font size: {config.render.font_size}")

## 2. Execution / 実行

Run the pipeline with the configuration above.

In [ ]:
# Run the pipeline
result = run_pipeline(config)

print(f"\nPipeline execution completed.")
print(f"Total images: {result.total_images}")
print(f"Parquet path: {result.parquet_path}")
print(f"Elapsed time: {result.elapsed_seconds:.2f}s")
print(f"Failed fonts: {len(result.failed_fonts)}")

## 3. Result Details / 結果の詳細

Inspect the results per font.

In [ ]:
# Results per font
for font_result in result.font_results:
    font_name = Path(font_result.font_path).stem
    print(f"\nFont: {font_name}")
    print(f"  Images written: {font_result.images_written}")
    print(f"  Charset skipped (no glyph): {len(font_result.charset_skipped)}")
    print(f"  Render skipped (overflow): {len(font_result.render_skipped)}")

## 4. Dataset Inspection / データセット確認

Inspect the generated Parquet file.

In [ ]:
import pyarrow.parquet as pq

# Load the Parquet file
parquet_path = result.parquet_path
if parquet_path.exists():
    table = pq.read_table(str(parquet_path))
    print(f"Schema:")
    print(table.schema)
    print(f"\nRecord count: {table.num_rows}")
    print(f"\nFirst 5 records:")
    records = table.to_pylist()
    for rec in records[:5]:
        print(f"  {rec['file']:<40s} char='{rec['char']}' font={Path(rec['font_path']).stem}")
else:
    print(f"Parquet file not found: {parquet_path}")

## 5. Output Structure / 出力構造

Inspect the generated file structure.

In [ ]:
# Check output directory structure
output_root = Path(config.output_dir)
if output_root.exists():
    print(f"Output directory: {output_root}")
    print(f"\nDirectory structure:")
    
    # Root files
    root_files = [f for f in output_root.iterdir() if f.is_file()]
    for f in sorted(root_files):
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<30s} {size_kb:>8.1f} KB")
    
    # Images directory
    images_dir = output_root / "images"
    if images_dir.exists():
        num_images = len(list(images_dir.glob("*.png")))
        print(f"\n  images/")
        print(f"    {num_images} PNG files")
        
        # Show first few filenames
        first_pngs = sorted(images_dir.glob("*.png"))[:5]
        for png in first_pngs:
            print(f"      {png.name}")
        if num_images > 5:
            print(f"      ... and {num_images - 5} more")
else:
    print(f"Output directory does not exist: {output_root}")

## 6. Configuration Variants / 設定のバリエーション

Different charset and rendering options.

In [ ]:
# Example 1: Uppercase letters with larger font
config_large = PipelineConfig(
    charset="uppercase",
    font_dir="./fonts",
    output_dir="./test_large_output",
    render=RenderConfig(font_size=72, image_size=(128, 128)),
    workers=2,
)

print("Example config: uppercase, large font (72px), large image (128x128)")
print(f"  Config created (not yet executed)")

# Example 2: Multiple charset specs (preset + literal)
config_mixed = PipelineConfig(
    charset=["digits", "+-*=/"],  # list of preset + literal
    font_dir="./fonts",
    output_dir="./test_mixed_output",
    workers=1,
)

print("\nExample config: mixed charset (digits + operators)")
print(f"  Config created (not yet executed)")

## Summary / まとめ

| Step | Method | Output |
|------|--------|--------|
| Configure | `PipelineConfig(charset, font_dir, ...)` | Configuration object |
| Execute | `run_pipeline(config)` | PipelineResult |
| Inspect | `result.font_results`, `result.parquet_path` | Metrics + output path |
| Read dataset | `pyarrow.parquet.read_table()` | Table of records |

**Key design points:**
- Config is the **single source of truth** — all parameters in one place
- Fonts are processed **in parallel** (ThreadPoolExecutor) but **deterministically** (sorted order)
- Errors are **handled gracefully** — one font failure doesn't stop the pipeline
- Output is **flat** — all images in `images/`, all metadata in Parquet
- Reproducibility is **guaranteed** — same config + fonts = same output